# zero-shot + few-shot




In [ ]:
ACTIVE_MODEL = "Phi-4-mini-instruct" 
DEFAULT_QUANT = "4bit"  
RUN_ZERO_SHOT = True
RUN_FEWSHOT = True
SMOKE_N = 0  


In [2]:
!pip install -q -U git+https://github.com/huggingface/transformers.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
!pip install accelerate datasets seqeval bitsandbytes

In [2]:
# !pip install -q -U "transformers>=4.49.0" accelerate datasets seqeval bitsandbytes

In [4]:
import torch
print(torch.cuda.is_available())  # должно быть True
print(torch.cuda.get_device_name(0))

True
NVIDIA L4


In [5]:
import gc
import torch

for name in ("model", "tokenizer", "trainer"):
    if name in globals():
        del globals()[name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("OK")

OK


In [6]:
torch.cuda.is_available()

True

In [ ]:
# !pip install -q "transformers>=4.47" accelerate datasets seqeval bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.3 MB/s eta 0:00:00


In [7]:
import gc
import json
import os
import re
from collections import Counter
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, classification_report as clf_report
from seqeval.metrics import (
    f1_score as seq_f1,
    precision_score as seq_precision,
    recall_score as seq_recall,
)
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("torch", torch.__version__, "cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


torch 2.10.0+cu128 cuda: True
NVIDIA L4


In [ ]:
from huggingface_hub import login, whoami

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if HF_TOKEN:
    login(token="вставьте свой")
    print("HF OK:", whoami()["name"])
else:
    login()  # интерактивно в браузере
    print("HF OK:", whoami()["name"])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


HF OK: zaryana2004


In [9]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/nlu_results"
except ImportError:
    SAVE_DIR = str(Path.cwd() / "nlu_results")

os.makedirs(SAVE_DIR, exist_ok=True)
print("SAVE_DIR:", SAVE_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SAVE_DIR: /content/drive/MyDrive/nlu_results


In [10]:
SYSTEM_PROMPT = """Ты — система понимания естественного языка (NLU).
Внимательно прочитай текст запроса пользователя, классифицируй его намерение (intent) и выдели все возможные слоты (slots), соответствующие этому намерению.

ВАЖНО: слова в слотах должны быть в той же морфологической форме, что и в тексте. Цифры записывай текстом.

---

ДОСТУПНЫЕ НАМЕРЕНИЯ И ИХ СЛОТЫ:

BookRestaurant — бронирование ресторана, кафе и т.п.:
  слоты: cuisine, datetime, facility, location, party_size_description,
          party_size_number, restaurant_name, restaurant_type, served_dish, sort

SearchScreeningEvent — поиск кинопоказов:
  слоты: datetime, location, movie_name, movie_type, object_location_type, object_type

SearchCreativeWork — поиск творческих произведений:
  слоты: object_name, object_type

AddToPlaylist — добавление в плейлист:
  слоты: artist, entity_name, music_item, playlist, reference

RateBook — оценка книги:
  слоты: best_rating, object_name, object_part_of_series_type, object_select,
          object_type, rating_unit, rating_value

PlayMusic — воспроизведение музыки:
  слоты: album, artist, datetime, genre, music_item, playlist, service, sort, track

weather/find — запрос погоды:
  слоты: condition_description, condition_temperature, datetime, location, weather/attribute

alarm/cancel_alarm, alarm/modify_alarm, alarm/set_alarm, alarm/show_alarms,
alarm/snooze_alarm, alarm/time_left_on_alarm — слоты: datetime, reference, recurring_datetime, reminder/todo

reminder/cancel_reminder, reminder/set_reminder, reminder/show_reminders — слоты: datetime, reference, reminder/todo

---

СТРОГИЕ ПРАВИЛА:
1. Используй ТОЛЬКО намерения и слоты из списка выше.
2. Копируй значения слотов ТОЧНО из текста пользователя.
3. Если слот в тексте отсутствует — не включай его в ответ.
4. Выведи ТОЛЬКО валидный JSON без комментариев.
5. НИКОГДА не придумывай новые значения слотов.

Формат ответа:
{"intent": "...", "slots": {"slot_name": "value from text"}}"""

INTENTS = [
    "AddToPlaylist", "BookRestaurant", "PlayMusic", "RateBook",
    "SearchCreativeWork", "SearchScreeningEvent",
    "alarm/cancel_alarm", "alarm/modify_alarm", "alarm/set_alarm",
    "alarm/show_alarms", "alarm/snooze_alarm", "alarm/time_left_on_alarm",
    "reminder/cancel_reminder", "reminder/set_reminder", "reminder/show_reminders",
    "weather/find",
]

SLOTS = [
    "alarm/alarm_modifier", "album", "artist", "best_rating",
    "condition_description", "condition_temperature", "cuisine", "datetime",
    "entity_name", "facility", "genre", "location", "movie_name", "movie_type",
    "music_item", "negation", "news/type", "object_location_type", "object_name",
    "object_part_of_series_type", "object_select", "object_type",
    "party_size_description", "party_size_number", "playlist", "rating_unit",
    "rating_value", "recurring_datetime", "reference", "reminder/reminder_modifier",
    "reminder/todo", "restaurant_name", "restaurant_type", "served_dish",
    "service", "sort", "timer/attributes", "track", "weather/attribute",
    "weather/temperatureUnit",
]
print("Промпт и схема готовы")


Промпт и схема готовы


In [11]:
def _resolve_conll(path):
    p = Path(path)
    return p if p.is_file() else Path.cwd() / path


def parse_conll_to_dicts(file_path, language="ru"):
    data = []
    words, bio = [], []
    cur_id = cur_intent = None
    text_ru = text_en = text_plain = None

    def pick_text():
        if language == "en":
            return text_plain or text_en or text_ru or ""
        return text_ru or text_plain or text_en or ""

    def flush_sentence():
        nonlocal words, bio, cur_id, cur_intent, text_ru, text_en, text_plain
        if not words:
            return
        slots = {}
        slot_name, buf = None, []
        for w, t in zip(words, bio):
            if t.startswith("B-"):
                if slot_name:
                    val = " ".join(buf)
                    if slot_name in slots:
                        slots[slot_name] += " | " + val
                    else:
                        slots[slot_name] = val
                slot_name = t[2:]
                buf = [w]
            elif t.startswith("I-") and slot_name == t[2:]:
                buf.append(w)
            else:
                if slot_name:
                    val = " ".join(buf)
                    if slot_name in slots:
                        slots[slot_name] += " | " + val
                    else:
                        slots[slot_name] = val
                slot_name, buf = None, []
        if slot_name:
            val = " ".join(buf)
            if slot_name in slots:
                slots[slot_name] += " | " + val
            else:
                slots[slot_name] = val
        data.append({
            "id": cur_id,
            "text": pick_text(),
            "intent": cur_intent or "none",
            "slots": slots,
            "original_bio": bio.copy(),
            "original_words": words.copy(),
        })
        words, bio = [], []
        cur_id = cur_intent = None
        text_ru = text_en = text_plain = None

    with open(_resolve_conll(file_path), encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line.startswith("# id:"):
                cur_id = line.split(":", 1)[1].strip()
            elif line.startswith("# text RU:"):
                text_ru = line.split(":", 1)[1].strip()
            elif line.startswith("# text EN:"):
                text_en = line.split(":", 1)[1].strip()
            elif line.startswith("# text ="):
                text_plain = line.split("=", 1)[1].strip()
            elif line.startswith("# text:"):
                text_plain = line.split(":", 1)[1].strip()
            elif line.startswith("# intent ="):
                cur_intent = line.split("=", 1)[1].strip()
            elif line.startswith("# intent:"):
                cur_intent = line.split(":", 1)[1].strip()
            elif line.startswith("#"):
                pass
            elif line == "":
                flush_sentence()
            else:
                parts = line.split("\t")
                if len(parts) >= 4:
                    words.append(parts[1])
                    bio.append(parts[-1])
    flush_sentence()
    return data


ru_train_data = parse_conll_to_dicts("ru.train.conll", language="ru")
test_data = parse_conll_to_dicts("ru.test.conll", language="ru")
ru_train_data = [x for x in ru_train_data if x["intent"] in INTENTS]
test_data = [x for x in test_data if x["intent"] in INTENTS]
if SMOKE_N > 0:
    test_data = test_data[:SMOKE_N]
print("ru.train:", len(ru_train_data), "test:", len(test_data))


ru.train: 36966 test: 532


In [12]:
INTENT_THRESHOLD = 500
SLOT_THRESHOLD = 500
IGNORED_INTENTS = {"weather/checkSunrise", "weather/checkSunset"}
intent_counter, slot_counter = Counter(), Counter()
for item in ru_train_data:
    if item["intent"] in IGNORED_INTENTS:
        continue
    intent_counter[item["intent"]] += 1
    for s in item["slots"]:
        slot_counter[s] += 1
WEAK_INTENTS = {i for i, c in intent_counter.items() if c < INTENT_THRESHOLD}
WEAK_SLOTS = {s for s, c in slot_counter.items() if c < SLOT_THRESHOLD}
print("weak intents:", len(WEAK_INTENTS), "weak slots:", len(WEAK_SLOTS))


weak intents: 3 weak slots: 17


In [13]:
MODEL_REGISTRY = {
    "Qwen2.5-3B-Instruct": {"hf_id": "Qwen/Qwen2.5-3B-Instruct", "family": "qwen"},
    "Qwen2.5-7B-Instruct": {"hf_id": "Qwen/Qwen2.5-7B-Instruct", "family": "qwen"},
    "google/gemma-2-2b-it": {"hf_id": "google/gemma-2-2b-it", "family": "gemma"},
    "google/gemma-2-9b-it": {"hf_id": "google/gemma-2-9b-it", "family": "gemma"},
    "Phi-4-mini-instruct": {"hf_id": "microsoft/Phi-4-mini-instruct", "family": "qwen"},
    "Mistral-7B-Instruct-v0.3": {"hf_id": "mistralai/Mistral-7B-Instruct-v0.3", "family": "qwen"},
}
for _k, _v in MODEL_REGISTRY.items():
    if "mode" not in _v:
        _v["mode"] = DEFAULT_QUANT

MODEL_NAME = ACTIVE_MODEL
CFG = MODEL_REGISTRY[ACTIVE_MODEL]

In [ ]:
def unload_model():
    global model, tokenizer
    if "model" in globals():
        del model
    if "tokenizer" in globals():
        del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("GPU cleared")


def _clear_phi_module_cache():
    import shutil
    base = Path.home() / ".cache/huggingface/modules/transformers_modules/microsoft"
    if not base.is_dir():
        return
    for p in base.iterdir():
        if p.is_dir() and "phi" in p.name.lower():
            shutil.rmtree(p, ignore_errors=True)


def _prepare_config(config):
    rs = getattr(config, "rope_scaling", None)
    if isinstance(rs, dict) and rs and "type" not in rs:
        config.rope_scaling = None
    theta = getattr(config, "rope_theta", None) or 1_000_000.0
    rp = getattr(config, "rope_parameters", None)
    if rp is None:
        config.rope_parameters = {"rope_type": "default", "rope_theta": theta}
    else:
        rp = dict(rp)
        rp.setdefault("rope_type", "default")
        rp.setdefault("rope_theta", theta)
        config.rope_parameters = rp
    return config


def load_model(registry_key: str):
    global model, tokenizer, MODEL_NAME, CFG
    unload_model()
    MODEL_NAME = registry_key
    CFG = MODEL_REGISTRY[registry_key]
    hf_id = CFG["hf_id"]

    is_giga = CFG.get("family") == "gigachat"
    is_phi = ("phi" in hf_id.lower()) and not is_giga
    if is_phi:
        _clear_phi_module_cache()
    remote = True if is_giga else (not is_phi)

    tok_kw = dict(trust_remote_code=remote)
    if HF_TOKEN:
        tok_kw["token"] = HF_TOKEN
    tokenizer = AutoTokenizer.from_pretrained(hf_id, **tok_kw)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    mode = CFG.get("mode", DEFAULT_QUANT)

    if is_giga:
        load_kw = dict(
            device_map="auto",
            trust_remote_code=True,
            low_cpu_mem_usage=True,
            max_memory={0: "22GiB"},
        )
        if HF_TOKEN:
            load_kw["token"] = HF_TOKEN
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
        )
        load_kw["quantization_config"] = bnb
        model = AutoModelForCausalLM.from_pretrained(hf_id, **load_kw)

    else:
        load_kw = dict(device_map="auto", trust_remote_code=remote, low_cpu_mem_usage=True)
        if HF_TOKEN:
            load_kw["token"] = HF_TOKEN

        if is_phi:
            load_kw["attn_implementation"] = "eager"
        else:
            cfg_kw = dict(trust_remote_code=remote)
            if HF_TOKEN:
                cfg_kw["token"] = HF_TOKEN
            config = _prepare_config(AutoConfig.from_pretrained(hf_id, **cfg_kw))
            load_kw["config"] = config

        if mode == "fp16":
            load_kw["torch_dtype"] = torch.float16
            model = AutoModelForCausalLM.from_pretrained(hf_id, **load_kw)
        elif mode == "4bit":
            bnb = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
            )
            load_kw["quantization_config"] = bnb
            model = AutoModelForCausalLM.from_pretrained(hf_id, **load_kw)
        else:
            raise ValueError(f"Unknown mode: {mode!r} (use fp16 or 4bit)")

    model.eval()
    print("Loaded", registry_key, "->", hf_id, "| remote:", remote, "| mode:", mode)
    return model, tokenizer


model, tokenizer = load_model(ACTIVE_MODEL)

In [15]:
def _model_family():
    fam = CFG.get("family", "qwen")
    mid = str(getattr(model.config, "_name_or_path", "")).lower()
    if "gemma" in mid:
        return "gemma"
    if "giga" in mid:
        return "gigachat"
    return fam


def _assistant_json(ex):
    return json.dumps({"intent": ex["intent"], "slots": ex["slots"]}, ensure_ascii=False)


def _build_messages_zeroshot(text):
    if _model_family() == "gemma":
        return [{"role": "user", "content": f"{SYSTEM_PROMPT}\n\nUser utterance:\n{text}"}]
    return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": text}]


def _build_messages_fewshot(text, few_shot_examples):
    if _model_family() == "gemma":
        msgs = [{"role": "user", "content": SYSTEM_PROMPT},
                {"role": "assistant", "content": "OK. JSON intent and slots."}]
        for ex in few_shot_examples:
            msgs.append({"role": "user", "content": ex["text"]})
            msgs.append({"role": "assistant", "content": _assistant_json(ex)})
        msgs.append({"role": "user", "content": text})
        return msgs
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}]
    for ex in few_shot_examples:
        msgs.append({"role": "user", "content": ex["text"]})
        msgs.append({"role": "assistant", "content": _assistant_json(ex)})
    msgs.append({"role": "user", "content": text})
    return msgs


def bio_to_spans(bio_tags, words):
    spans = []
    current_slot, current_tokens = None, []
    for word, tag in zip(words, bio_tags):
        if tag.startswith("B-"):
            if current_slot:
                spans.append((current_slot, " ".join(current_tokens).rstrip(".,!?;:")))
            current_slot = tag[2:]
            current_tokens = [word]
        elif tag.startswith("I-") and current_slot == tag[2:]:
            current_tokens.append(word)
        else:
            if current_slot:
                spans.append((current_slot, " ".join(current_tokens).rstrip(".,!?;:")))
            current_slot, current_tokens = None, []
    if current_slot:
        spans.append((current_slot, " ".join(current_tokens).rstrip(".,!?;:")))
    return set(spans)


def _parse_output(text, confidence=1.0, confidence_threshold=0.5):
    try:
        j = re.search(r"\{.*\}", text, re.DOTALL)
        if not j:
            return {"intent": "none", "slots": {}, "confidence": confidence, "weak_intent": True}
        result = json.loads(j.group())
    except Exception:
        return {"intent": "none", "slots": {}, "confidence": confidence, "weak_intent": True}
    pred_intent = result.get("intent")
    if not isinstance(pred_intent, str) or not pred_intent.strip():
        pred_intent = "none"
    matched = next((i for i in INTENTS if i.lower() == pred_intent.lower()), pred_intent)
    result["intent"] = matched
    result["confidence"] = confidence
    result["weak_intent"] = confidence < confidence_threshold
    slots = result.get("slots", {})
    if isinstance(slots, list):
        merged = {}
        for item in slots:
            if isinstance(item, dict):
                merged.update(item)
        result["slots"] = merged
    elif not isinstance(slots, dict):
        result["slots"] = {}
    return result


def _generate(messages, confidence_threshold=0.5):
    for msg in messages:
        if not msg.get("content"):
            return {"intent": "none", "slots": {}, "confidence": 0.0, "weak_intent": True}
    if _model_family() == "gigachat":
        input_ids = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt"
        ).to(model.device)
        with torch.no_grad():
            out = model.generate(
                input_ids, max_new_tokens=120, do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        gen_text = tokenizer.decode(out[0, input_ids.shape[1]:], skip_special_tokens=True).strip()
        return _parse_output(gen_text, confidence=1.0, confidence_threshold=confidence_threshold)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=120, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            output_scores=True, return_dict_in_generate=True,
        )
    input_len = inputs["input_ids"].shape[1]
    gen_sequences = outputs.sequences[:, input_len:]
    if gen_sequences.shape[1] == 0 or not outputs.scores:
        return {"intent": "none", "slots": {}, "confidence": 0.0, "weak_intent": True}
    gen_scores = torch.stack(outputs.scores, dim=1)
    probs = torch.softmax(gen_scores, dim=-1)
    token_probs = torch.gather(probs, 2, gen_sequences.unsqueeze(-1)).squeeze(-1)
    confidence = round(float(token_probs.mean().item()), 4)
    gen_text = tokenizer.decode(gen_sequences[0], skip_special_tokens=True).strip()
    if not gen_text or "{" not in gen_text:
        return {"intent": "none", "slots": {}, "confidence": 0.0, "weak_intent": True}
    return _parse_output(gen_text, confidence=confidence, confidence_threshold=confidence_threshold)


def generate_prediction_zeroshot(text, confidence_threshold=0.5):
    return _generate(_build_messages_zeroshot(text), confidence_threshold)


def generate_prediction_fewshot(text, few_shot_examples, confidence_threshold=0.5):
    return _generate(_build_messages_fewshot(text, few_shot_examples), confidence_threshold)

print("infer OK")


infer OK


In [16]:
def make_experiment_name(experiment_type, intents):
    part = "__".join(sorted(intents)).replace("/", "_")
    return f"{experiment_type}__{part}"


def get_fewshot_examples(target_intents):
    examples = []
    for target_intent in target_intents:
        candidates = sorted(
            [x for x in ru_train_data if x["intent"] == target_intent],
            key=lambda x: (len(x["slots"]), len(x["text"])),
            reverse=True,
        )
        if candidates:
            examples.append(candidates[0])
    return examples


FEWSHOT_CONFIGS = {
    "few_shot_1_popular": ["weather/find"],
    "few_shot_1_problem": ["SearchScreeningEvent"],
    "few_shot_1_slots": ["BookRestaurant"],
    "few_shot_5": [
        "SearchScreeningEvent", "BookRestaurant", "weather/find",
        "alarm/set_alarm", "reminder/set_reminder",
    ],
    "few_shot_10": [
        "weather/find", "alarm/set_alarm", "alarm/snooze_alarm", "alarm/cancel_alarm",
        "reminder/set_reminder", "BookRestaurant", "PlayMusic",
        "SearchCreativeWork", "SearchScreeningEvent", "AddToPlaylist",
    ],
}
print("few-shot configs:", list(FEWSHOT_CONFIGS))


few-shot configs: ['few_shot_1_popular', 'few_shot_1_problem', 'few_shot_1_slots', 'few_shot_5', 'few_shot_10']


In [17]:
def run_evaluation(predict_fn, experiment_type, model_name=None, fewshot_examples=None):
    model_name = model_name or MODEL_NAME
    exp_dir = f"{SAVE_DIR}/{experiment_type}/{model_name}"
    os.makedirs(exp_dir, exist_ok=True)

    if fewshot_examples is not None:
        with open(f"{exp_dir}/fewshot_examples.json", "w", encoding="utf-8") as f:
            json.dump(
                [{"intent": ex["intent"], "text": ex["text"], "slots": ex["slots"]}
                 for ex in fewshot_examples],
                f, ensure_ascii=False, indent=2,
            )

    csv_path = f"{exp_dir}/results.csv"
    log_path = f"{exp_dir}/log.txt"
    checkpoint_path = f"{exp_dir}/checkpoint.json"

    if (Path(exp_dir) / "metrics_summary.json").is_file() and not Path(checkpoint_path).is_file():
        print(f"SKIP (готово): {experiment_type}/{model_name}")
        return

    results_log = []
    all_true_bio, all_pred_bio = [], []
    all_true_bio_seq, all_pred_bio_seq = [], []
    start_idx = 0
    log_lines = []

    def log(msg):
        print(msg)
        log_lines.append(str(msg))

    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, "r", encoding="utf-8") as f:
            checkpoint = json.load(f)
        results_log = checkpoint["results_log"]
        all_true_bio = [set(tuple(x) for x in s) for s in checkpoint["all_true_bio"]]
        all_pred_bio = [set(tuple(x) for x in s) for s in checkpoint["all_pred_bio"]]
        all_true_bio_seq = checkpoint.get("all_true_bio_seq", [])
        all_pred_bio_seq = checkpoint.get("all_pred_bio_seq", [])
        start_idx = checkpoint["next_idx"]
        print(f"Найден чекпоинт. Продолжаем с примера {start_idx}")
    else:
        print("Чекпоинт не найден, начинаем с нуля")

    for idx, item in enumerate(tqdm(
        test_data[start_idx:],
        desc=f"Тестируем {experiment_type}/{model_name}",
        unit="doc",
        initial=start_idx,
        total=len(test_data),
    )):
        real_idx = idx + start_idx

        pred_json = predict_fn(item["text"])
        pred_intent = pred_json.get("intent", "none")
        pred_slots = pred_json.get("slots", {})
        confidence = pred_json.get("confidence", 0.0)
        is_correct = pred_intent == item["intent"]

        text_words = item.get("original_words") or (item["text"].split() if item["text"] else [])
        true_spans = bio_to_spans(item["original_bio"], text_words)

        pred_spans = set()
        for slot_name, slot_value in pred_slots.items():
            if slot_name in SLOTS and slot_value is not None:
                slot_value_clean = str(slot_value).strip().rstrip(".,!?;:")
                if slot_value_clean.lower() not in item["text"].lower():
                    continue
                pred_spans.add((slot_name, slot_value_clean))

        all_true_bio.append(true_spans)
        all_pred_bio.append(pred_spans)

        true_bio_seq = item["original_bio"] if item["original_bio"] else ["O"] * len(text_words)
        pred_bio_seq = ["O"] * len(text_words)
        for slot_name, slot_value in pred_spans:
            slot_tokens = slot_value.split()
            for i in range(len(text_words) - len(slot_tokens) + 1):
                if [w.lower() for w in text_words[i:i + len(slot_tokens)]] == [
                    w.lower() for w in slot_tokens
                ]:
                    pred_bio_seq[i] = f"B-{slot_name}"
                    for j in range(1, len(slot_tokens)):
                        pred_bio_seq[i + j] = f"I-{slot_name}"
                    break

        all_true_bio_seq.append(true_bio_seq)
        all_pred_bio_seq.append(pred_bio_seq)

        is_weak_intent = item["intent"] in WEAK_INTENTS
        is_weak_slot = any(s in WEAK_SLOTS for s in item["slots"].keys())

        log(f"\n[ID: {item['id']}] [{real_idx + 1}/{len(test_data)}]")
        log(f"  Текст:          {item['text']}")
        log(f"  Истинный интент:{item['intent']}")
        log(f"  Предсказанный:  {pred_intent}  {'✓' if is_correct else '✗'}")
        log(f"  Confidence:     {confidence}  {'⚠ СЛАБЫЙ' if pred_json.get('weak_intent') else ''}")
        log(f"  Истинные слоты: {item['slots']}")
        log(f"  Предсказанные:  {pred_slots}")
        log(f"  Слабый интент:  {is_weak_intent}")
        log(f"  Слабые слоты:   {is_weak_slot}")

        results_log.append({
            "id": item["id"],
            "text": item["text"],
            "true_intent": item["intent"],
            "pred_intent": pred_intent,
            "intent_correct": is_correct,
            "confidence": confidence,
            "weak_by_conf": pred_json.get("weak_intent"),
            "weak_intent_real": is_weak_intent,
            "weak_slot_real": is_weak_slot,
            "true_slots": str(item["slots"]),
            "pred_slots": str(pred_slots),
        })

        if (real_idx + 1) % 10 == 0:
            checkpoint = {
                "results_log": results_log,
                "all_true_bio": [list(s) for s in all_true_bio],
                "all_pred_bio": [list(s) for s in all_pred_bio],
                "all_true_bio_seq": all_true_bio_seq,
                "all_pred_bio_seq": all_pred_bio_seq,
                "next_idx": real_idx + 1,
            }
            with open(checkpoint_path, "w", encoding="utf-8") as f:
                json.dump(checkpoint, f, ensure_ascii=False)
            pd.DataFrame(results_log).to_csv(csv_path, index=False, encoding="utf-8-sig")
            print(f"  [чекпоинт сохранён: {real_idx + 1}/{len(test_data)}]")

    log(f"\n{'=' * 50}")
    log(f" ОТЧЕТ: {experiment_type.upper()} / {model_name} ")
    log(f"{'=' * 50}")

    valid_results = [r for r in results_log if r["true_intent"] in INTENTS]
    true_intents = [r["true_intent"] for r in valid_results]
    pred_intents = [r["pred_intent"] for r in valid_results]

    intent_accuracy = (
        sum(r["pred_intent"] == r["true_intent"] for r in valid_results) / len(valid_results)
        if valid_results
        else 0.0
    )
    intent_f1_macro = f1_score(true_intents, pred_intents, average="macro", zero_division=0)
    intent_f1_weighted = f1_score(true_intents, pred_intents, average="weighted", zero_division=0)

    log(f"\nIntent Accuracy:                {intent_accuracy:.4f}")
    log(f"Intent F1 macro:                {intent_f1_macro:.4f}")
    log(f"Intent F1 weighted:             {intent_f1_weighted:.4f}")
    log("\nIntent Classification Report:")
    report_labels = [i for i in INTENTS if i in true_intents or i in pred_intents]
    log(clf_report(true_intents, pred_intents, labels=report_labels, zero_division=0))

    weak_conf = sum(1 for r in results_log if r.get("weak_by_conf"))
    weak_intent_real = sum(1 for r in results_log if r.get("weak_intent_real"))
    weak_slot_real = sum(1 for r in results_log if r.get("weak_slot_real"))

    log(f"Слабых по уверенности (< 0.5):     {weak_conf} / {len(results_log)}")
    log(f"Слабых интентов (реально редкие):  {weak_intent_real} / {len(results_log)}")
    log(f"Слабых слотов (реально редкие):    {weak_slot_real} / {len(results_log)}")

    log("\nМетрики слотов (token-level, span-matching):")
    tp = fp = fn = 0
    for true_set, pred_set in zip(all_true_bio, all_pred_bio):
        tp += len(true_set & pred_set)
        fp += len(pred_set - true_set)
        fn += len(true_set - pred_set)

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

    log(f"Slot Precision: {precision:.4f}")
    log(f"Slot Recall:    {recall:.4f}")
    log(f"Slot F1:        {f1:.4f}")

    log("\nМетрики слотов (span-level, seqeval):")
    log(f"Slot Precision: {seq_precision(all_true_bio_seq, all_pred_bio_seq):.4f}")
    log(f"Slot Recall:    {seq_recall(all_true_bio_seq, all_pred_bio_seq):.4f}")
    log(f"Slot F1:        {seq_f1(all_true_bio_seq, all_pred_bio_seq):.4f}")

    log("\n===== PER SLOT (seqeval BIO) =====")
    from seqeval.metrics import classification_report as seq_slot_report

    slot_report = seq_slot_report(all_true_bio_seq, all_pred_bio_seq, output_dict=True)
    per_slot_rows = []
    for label, m in slot_report.items():
        if label in ("micro avg", "macro avg", "weighted avg", "accuracy"):
            continue
        per_slot_rows.append({
            "slot": label,
            "precision": round(m["precision"], 4),
            "recall": round(m["recall"], 4),
            "f1": round(m["f1-score"], 4),
            "support": int(m["support"]),
        })
    per_slot_bio_df = pd.DataFrame(per_slot_rows).sort_values("f1", ascending=False)
    per_slot_bio_path = f"{exp_dir}/per_slot_bio.csv"
    per_slot_bio_df.to_csv(per_slot_bio_path, index=False, encoding="utf-8-sig")
    log(per_slot_bio_df.to_string(index=False))
    log(f"Per-slot BIO CSV: {per_slot_bio_path}")

    log("\n===== PER SLOT (strict span, type+value) =====")
    slot_types = set()
    for spans in all_true_bio + all_pred_bio:
        for name, _ in spans:
            slot_types.add(name)
    per_slot_span_rows = []
    for slot in sorted(slot_types):
        tp_s = fp_s = fn_s = 0
        for true_set, pred_set in zip(all_true_bio, all_pred_bio):
            true_s = {(n, v) for n, v in true_set if n == slot}
            pred_s = {(n, v) for n, v in pred_set if n == slot}
            tp_s += len(true_s & pred_s)
            fp_s += len(pred_s - true_s)
            fn_s += len(true_s - pred_s)
        p = tp_s / (tp_s + fp_s) if (tp_s + fp_s) else 0.0
        r = tp_s / (tp_s + fn_s) if (tp_s + fn_s) else 0.0
        f1s = 2 * p * r / (p + r) if (p + r) else 0.0
        per_slot_span_rows.append({
            "slot": slot,
            "precision": round(p, 4),
            "recall": round(r, 4),
            "f1": round(f1s, 4),
            "support": tp_s + fn_s,
        })
    per_slot_span_df = pd.DataFrame(per_slot_span_rows).sort_values("f1", ascending=False)
    per_slot_span_path = f"{exp_dir}/per_slot_span.csv"
    per_slot_span_df.to_csv(per_slot_span_path, index=False, encoding="utf-8-sig")
    log(per_slot_span_df.to_string(index=False))
    log(f"Per-slot span CSV: {per_slot_span_path}")

    metrics_summary = {
        "experiment_type": experiment_type,
        "model_name": model_name,
        "n_samples": len(valid_results),
        "intent_accuracy": round(intent_accuracy, 4),
        "intent_f1_macro": round(intent_f1_macro, 4),
        "intent_f1_weighted": round(intent_f1_weighted, 4),
        "slot_precision_span": round(float(precision), 4),
        "slot_recall_span": round(float(recall), 4),
        "slot_f1_span": round(float(f1), 4),
        "slot_precision_seqeval": round(float(seq_precision(all_true_bio_seq, all_pred_bio_seq)), 4),
        "slot_recall_seqeval": round(float(seq_recall(all_true_bio_seq, all_pred_bio_seq)), 4),
        "slot_f1_seqeval": round(float(seq_f1(all_true_bio_seq, all_pred_bio_seq)), 4),
    }
    with open(f"{exp_dir}/metrics_summary.json", "w", encoding="utf-8") as mf:
        json.dump(metrics_summary, mf, ensure_ascii=False, indent=2)
    log(f"metrics_summary.json -> {exp_dir}/metrics_summary.json")

    pd.DataFrame(results_log).to_csv(csv_path, index=False, encoding="utf-8-sig")
    log(f"CSV сохранён в {csv_path}")

    with open(log_path, "w", encoding="utf-8") as f:
        f.write("\n".join(log_lines))
    log(f"Лог сохранён в {log_path}")

    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)
        print("Чекпоинт удалён — всё завершено")

    print("metrics_summary.json saved")
    return metrics_summary


print("Функция оценки готова")

Функция оценки готова


## Smoke + прогон

In [ ]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))
%timeit generate_prediction_zeroshot(test_data[0]["text"])

True NVIDIA L4
4.54 s ± 22.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
if RUN_ZERO_SHOT:
    run_evaluation(generate_prediction_zeroshot, "zero_shot", model_name=MODEL_NAME)


In [ ]:
if RUN_FEWSHOT:
    for exp_type, intents in FEWSHOT_CONFIGS.items():
        examples = get_fewshot_examples(intents)
        run_evaluation(
            lambda text, ex=examples: generate_prediction_fewshot(text, ex),
            exp_type,
            model_name=MODEL_NAME,
            fewshot_examples=examples,
        )


In [ ]:
import glob
from pathlib import Path

rows = []
for path in sorted(glob.glob(f"{SAVE_DIR}/*/{MODEL_NAME}/metrics_summary.json")):
    with open(path, encoding="utf-8") as f:
        m = json.load(f)
    m["experiment"] = Path(path).relative_to(SAVE_DIR).parts[0]
    rows.append(m)

if rows:
    display(pd.DataFrame(rows))
    safe_name = MODEL_NAME.replace("/", "_")
    out = f"{SAVE_DIR}/summary_{safe_name}.csv"
    pd.DataFrame(rows).to_csv(out, index=False)
    print("Saved:", out)
else:
    print("Нет метрик для", MODEL_NAME)